# Exercise 2: Sampling lab

**Lecture segment:** 3, the next-token distribution and sampling.
**Goal:** explain how temperature and top-p reshape the next-token distribution, and what that does to the generated text.
**Models:** OLMo 2 1B (base), runs locally. No API key needed.
**Time:** about 8 minutes.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")  # the shared helpers live in ../llm_helpers.py
from llm_helpers import *
import matplotlib.pyplot as plt

base = load(BASE)

## Task 1: Temperature reshapes the distribution

Temperature divides the model's raw scores (logits) before they are turned into probabilities.
Run the cell, then change the prompt and the temperatures and run it again.

In [ ]:
prompt = "My favourite food is"  # TODO: try a prompt of your own
temperatures = [0.3, 1.0, 2.0]   # TODO: try other values

logits = base(encode(base, prompt)).logits[0, -1].float().detach()
fig, axes = plt.subplots(1, len(temperatures), figsize=(14, 3.5), sharey=True)
for ax, T in zip(axes, temperatures):
    p, ids = (logits / T).softmax(-1).topk(12)
    ax.bar([base.tok.decode(i) for i in ids.tolist()], p.cpu())
    ax.set_yscale("log")
    ax.set_ylim(1e-4, 1)
    ax.set_title(f"temperature = {T}: top 12 hold {p.sum():.0%}")
    ax.tick_params(axis="x", rotation=60)
axes[0].set_ylabel("probability (log scale)")
plt.tight_layout()

**Observe:** The order of the tokens never changes. What does change? Where does the probability go that is not in the top 12?

*Your answer:*

## Task 2: Temperature in generated text

The same prompt at several temperatures, with two random seeds each. `temperature=0` means greedy: always the most likely token.

In [ ]:
prompt = "Once upon a time, in a small village in the Alps,"
temperatures = [0.0, 0.7, 1.5]  # TODO: find the temperature where the text starts to fall apart

for T in temperatures:
    print(f"--- temperature {T}")
    for seed in [1, 2]:
        print(repr(complete(base, prompt, max_new_tokens=30, temperature=T, seed=seed)))

**Observe:** Why are both seeds identical at temperature 0, and different at higher temperatures?

*Your answer:*

## Task 3: top-p cuts off the tail

top-p (nucleus sampling) keeps only the most likely tokens whose probabilities add up to p, and samples among those.
Here the temperature is high on purpose.

In [ ]:
top_ps = [1.0, 0.9, 0.5]  # TODO: try other values between 0 and 1

for p in top_ps:
    print(f"top_p={p}:", repr(complete(base, prompt, max_new_tokens=30, temperature=1.5, top_p=p, seed=1)))

**Observe:** Why does a lower top-p rescue the text at high temperature? Chat services often use a temperature around 0.7 with top-p around 0.9: what does each setting do?

*Your answer:*

## Wrap-up

In one or two sentences: what does a chat interface's "temperature" setting do to the next-token distribution?

*Your answer:*